# importing dataset

In [1]:
!mkdr -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

/bin/bash: line 1: mkdr: command not found
cp: cannot stat 'kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d mohammadhossein77/brain-tumors-dataset

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [ ]:
!pip install patool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 4.7 MB/s eta 0:00:00


#unzipping and extracting the dataset in new folder

In [ ]:
import patoolib
patoolib.extract_archive("/content/brain-tumors-dataset.zip")

PatoolError: file `/content/brain-tumors-dataset.zip' was not found

#importing modules

In [ ]:
import io
import os
import cv2
import keras
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from keras.models import Sequential
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split
from keras.layers import Dense,Flatten,Conv2D,MaxPooling2D,Dropout









In [ ]:
from tensorflow.keras.preprocessing import image
img=image.load_img('/content/Data/Tumor/pituitary_tumor/P_150.jpg',target_size=(150,150))
plt.imshow(img)

#image preprocessing


In [ ]:
x=[]
y=[]
a=0
b=0
c=0
d=0
ims=150
tu=['glioma_tumor','meningioma_tumor','pituitary_tumor']
no=['Normal']
normal=os.path.join('/content/Data/Normal')
for j in os.listdir(normal):
  img=cv2.imread(os.path.join(normal,j))
  img=cv2.resize(img,(ims,ims))
  x.append(img)
  y.append('normal')
  a=a+1

for i in tu:
  tumor=os.path.join('/content/Data/Tumor',i)
  for j in os.listdir(tumor):
    img=cv2.imread(os.path.join(tumor,j))
    img=cv2.resize(img,(ims,ims))
    x.append(img)
    y.append(i)
    if i=='glioma_tumor':
      b=b+1
    elif i=='meningioma_tumor':
      c=c+1
    else:
      d=d+1



In [ ]:
import os
from collections import Counter

# Path to your dataset
dataset_path = '/content/Data'

# Categories and subcategories
categories = ['Tumor', 'Normal']
tumor_subcategories = ['glioma_tumor', 'pituitary_tumor', 'meningioma_tumor']

# Initialize a Counter to count images
image_count = Counter()

# Count images in Tumor Images subfolder
for subcategory in tumor_subcategories:
    subcategory_path = os.path.join(dataset_path, 'Tumor', subcategory)
    image_count[subcategory] = len(os.listdir(subcategory_path))

# Count images in Normal Images subfolder
normal_images_path = os.path.join(dataset_path, 'Normal')
image_count['Normal Images'] = len(os.listdir(normal_images_path))

# Display the counts
print(image_count)


In [ ]:
import matplotlib.pyplot as plt

# Labels and counts
labels = list(image_count.keys())
counts = list(image_count.values())

# Plotting the distribution
plt.figure(figsize=(10, 6))
plt.bar(labels, counts, color=['lightblue', 'lightsteelblue', 'steelblue', 'skyblue'])
plt.title('DISTRIBUTION OF MRI IMAGES',size=15)
plt.xlabel('Categories',size=12,font='lexend')
plt.ylabel('Number of images',size=12,font='lexend')
plt.show()


#converting to numpy array

In [ ]:
x=np.array(x)
y=np.array(y)

#shuffling

In [ ]:
x,y=shuffle(x,y,random_state=101)

#train test split

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=101)


In [ ]:
ytrn=[]
for i in ytrain:
  if i=='glioma_tumor':
    ytrn.append(0)

  elif i=='meningioma_tumor':
    ytrn.append(1)
  elif i=='pituitary_tumor':
    ytrn.append(2)
  else:
    ytrn.append(3)

ytrain=ytrn
ytrain=tf.keras.utils.to_categorical(ytrain)

In [ ]:
yten=[]
for i in ytest:
  if i=='glioma_tumor':
    yten.append(0)
  elif i=='meningioma_tumor':
    yten.append(1)
  elif i=='pituitary_tumor':
    yten.append(2)
  else:
    yten.append(3)

ytest=yten
yte=tf.keras.utils.to_categorical(ytest)

#model creation

In [ ]:
model=Sequential()
model.add(Conv2D(32,(3,3),activation='relu',input_shape=(150,150,3)))
model.add(Conv2D(32,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))
model.add(Conv2D(64,(3,3),activation='relu'))
model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))
model.add(Conv2D(128,(3,3),activation='relu'))
model.add(Conv2D(128,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))
model.add(Conv2D(256,(3,3),activation='relu'))
model.add(Conv2D(256,(3,3),activation='relu'))
model.add(Flatten())
model.add(Dense(512,activation='relu'))
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4,activation='softmax'))

In [ ]:
model.summary( )

In [ ]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

#training

In [ ]:
model.fit(xtrain,ytrain,epochs=10,validation_split=(0.1))

In [ ]:
model.save('brain_tumor.keras')

#testing

In [ ]:
ypred=model.predict(xtest)

# Convert predictions to 1D array of predicted classes
ypred_classes = np.argmax(ypred,axis=1)

df=pd.DataFrame({'Actual':ytest,'Predicted':ypred_classes})
df

#testing with new image

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
img=image.load_img('/content/Data/Normal/N_1.jpg',target_size=(150,150))
plt.imshow(img)
ai=cv2.imread(f"/content/Data/Normal/N_1.jpg")
ai=cv2.resize(ai,(150,150))
ai=np.array(ai)
ai=ai.reshape((1,150,150,3))
a=model.predict(ai)
index=a.argmax()
if(index==0):
  print('glioma')
elif(index==1):
  print('meningioma')
elif(index==2):
  print('pituitary')
else:
  print('normal')

#graph plottting

In [ ]:
import matplotlib.pyplot as plt

# Assuming 'ypred_classes' contains predicted class labels and 'ytest' contains true class labels
# Count occurrences of each class
unique_predicted, counts_predicted = np.unique(ypred_classes, return_counts=True)
unique_true, counts_true = np.unique(ytest, return_counts=True)

# Create bar plots
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(unique_predicted, counts_predicted)
plt.xlabel('Predicted Classes')
plt.ylabel('Count')
plt.title('Distribution of Predicted Classes')

plt.subplot(1, 2, 2)
plt.plot(unique_true, counts_true)
plt.xlabel('True Classes')
plt.ylabel('Count')
plt.title('Distribution of True Classes')

plt.tight_layout()
plt.show()